In [4]:
from dotenv import load_dotenv
load_dotenv()


True

In [31]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate

loading the data

In [42]:
loader = PyPDFLoader("../data/Electricity (Prashant Kirad).pdf")
docs = loader.load()
len(docs)

86

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap = 200)
splitted_data = splitter.split_documents(docs)
len(splitted_data)

4

In [10]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings
)

d:\CODING\GENAI_Series\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\CODING\GENAI_Series\env\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SHIVAM\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate deve

In [15]:
query = "how much strong problem solving skill he has?"

In [48]:
data = vector_store.similarity_search(query=query)
context = ""
for doc in data:
    context += doc.page_content + "\n"

# print(context)

In [26]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")
res = llm.invoke(f"can you provide me the answer based on provided context for my question, context: {context} and question: {query}")
# print(res.text)

#### Chain - context generator / prompt / llm / strparser

In [47]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data:
        context += doc.page_content + "\n"
    
    return {
        "context" : context,
        "question" : query,
    }

In [46]:
prompt = PromptTemplate.from_template("""
    you are a helpful assistant and provide answered based on the context for user question. if you dont't know the answer , then you can say that 'i don't know.'
    context: {context}
    question: {question}
""")

In [33]:
rag_chain = get_context | prompt | llm 

In [50]:
res = rag_chain.invoke("Summarise his extra curricular skills ? ") 
print(res.text)


Based on the provided context, his extra-curricular activities and leadership skills include:

* **Leadership & Coordination:** Served as a Student Coordinator for the PARTH Society.
* **Event & Logistics Management:** Organized 5 educational trips and managed logistics for more than 100 participants.
* **Session Facilitation & Engagement:** Conducted over 50 sessions focused on mental health and spirituality to enhance student engagement.
